# Finetuning ViT Umur

In [19]:
import utils as u
import pandas as pd
from sklearn.model_selection import train_test_split
import tensorflow as tf
import keras
from keras import layers
from keras.utils import to_categorical
from PIL import Image
import numpy as np
from transformers import AutoImageProcessor, AutoModelForImageClassification, TFAutoModelForImageClassification
from transformers import TFAutoModel
from keras import layers, Model, Input
from keras.callbacks import ReduceLROnPlateau, ModelCheckpoint
import torch
from tqdm import tqdm
import os

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model_path = "dima806/facial_age_image_detection"
dir_path = "models/vit_age/"
os.makedirs(dir_path, exist_ok=True)

device, model_path

(device(type='cpu'), 'dima806/facial_age_image_detection')

## DemogPairs

### Load Dataset

In [6]:
data = u.load_demogpairs()
df = pd.DataFrame(data)
_dtable = u.display_table(data, n_items=[5, 5], hidden_columns=['db_code', 'image_path'])

full_path,label,label_idx
dataset/demogpairs/images/able_wanamakok/002.jpg,Asian_Females,5
dataset/demogpairs/images/able_wanamakok/004.jpg,Asian_Females,5
dataset/demogpairs/images/able_wanamakok/007.jpg,Asian_Females,5
dataset/demogpairs/images/able_wanamakok/008.jpg,Asian_Females,5
dataset/demogpairs/images/able_wanamakok/012.jpg,Asian_Females,5
...,...,...
dataset/demogpairs/images/zachary_quinto/177.jpg,White_Males,3
dataset/demogpairs/images/zachary_quinto/214.jpg,White_Males,3
dataset/demogpairs/images/zachary_quinto/217.jpg,White_Males,3
dataset/demogpairs/images/zachary_quinto/218.jpg,White_Males,3


## Split Data

In [7]:
# Split train/test
train_df, test_df = train_test_split(
    df, 
    test_size=0.2, 
    stratify=df["label_idx"], 
    random_state=42
)

num_classes = df["label_idx"].nunique()
num_classes

6

In [11]:
processor = AutoImageProcessor.from_pretrained(model_path)

def load_and_preprocess(path, label):
    image = Image.open(path).convert("RGB")
    encoding = processor(image, return_tensors="np")
    return encoding["pixel_values"][0], label

def make_dataset(df):
    X, y = [], []
    for _, row in tqdm(list(df.iterrows())):
        img, lbl = load_and_preprocess(row["full_path"], row["label_idx"])
        X.append(img)
        y.append(lbl)
    X = np.array(X, dtype="float32")
    y = to_categorical(y, num_classes=num_classes)
    return X, y

X_train, y_train = make_dataset(train_df)
X_test, y_test = make_dataset(test_df)

100%|█████████████████████████████████████████████████████████████████████████████| 2160/2160 [00:12<00:00, 172.48it/s]


In [14]:
def build_model(model_path, model_prefix_name='', lr=5e-5, optimizer_name="adam", dropout_rate=0.3, dense_units=256):
    # Load backbone
    base_model = TFAutoModel.from_pretrained(model_path)
    base_model.trainable = True

    # Input sesuai shape image yang diproses oleh processor
    inputs = Input(shape=X_train.shape[1:])

    # Ambil [CLS] token embedding dari backbone
    x = base_model(inputs)[0][:, 0, :]   # shape [batch, hidden_size]

    # Custom classification head
    x = layers.Dropout(dropout_rate)(x)
    x = layers.Dense(dense_units, activation="relu")(x)
    outputs = layers.Dense(num_classes, activation="softmax")(x)
    
    # Buat nama model unik
    model_name = f"{model_prefix_name}lr-{lr}_opt-{optimizer_name}_drop-{dropout_rate}_dense-{dense_units}"

    model = Model(inputs, outputs, name=model_name)

    # Optimizer
    if optimizer_name == "adam":
        optimizer = keras.optimizers.Adam(learning_rate=lr)
    elif optimizer_name == "adamw":
        optimizer = keras.optimizers.experimental.AdamW(learning_rate=lr, weight_decay=1e-4)
    elif optimizer_name == "sgd":
        optimizer = keras.optimizers.SGD(learning_rate=lr, momentum=0.9)

    model.compile(
        optimizer=optimizer,
        loss=keras.losses.CategoricalCrossentropy(),
        metrics=["accuracy"]
    )

    return model_name, model

build_model

<function __main__.build_model(model_path, model_prefix_name='', lr=5e-05, optimizer_name='adam', dropout_rate=0.3, dense_units=256)>

In [17]:
keras.backend.clear_session()
model_name, model = build_model(model_path, 'vit-age_', lr=0.0001, optimizer_name="adamw", dropout_rate=0.3, dense_units=512)
model.summary()
model_name, model

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFViTModel: ['classifier.weight', 'classifier.bias']
- This IS expected if you are initializing TFViTModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFViTModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
Some weights or buffers of the TF 2.0 model TFViTModel were not initialized from the PyTorch model and are newly initialized: ['vit.pooler.dense.weight', 'vit.pooler.dense.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model: "vit-age_lr-0.0001_opt-adamw_drop-0.3_dense-512"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 3, 224, 224)]     0         
                                                                 
 tf_vi_t_model (TFViTModel)  TFBaseModelOutputWithPo   86389248  
                             oling(last_hidden_state             
                             =(None, 197, 768),                  
                              pooler_output=(None, 7             
                             68),                                
                              hidden_states=None, at             
                             tentions=None)                      
                                                                 
 tf.__operators__.getitem (  (None, 768)               0         
 SlicingOpLambda)                                                
                    

('vit-age_lr-0.0001_opt-adamw_drop-0.3_dense-512',
 <keras.src.engine.functional.Functional at 0x224252023e0>)

In [ ]:
callbacks = [
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, verbose=1, min_lr=1e-7),
    ModelCheckpoint(
        filepath=f"{dir_path}/{model_name}_epoch-{{epoch:02d}}_valacc-{{val_accuracy:.4f}}_valloss-{{val_loss:.4f}}.keras",
        save_best_only=False,
        verbose=1
    )
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=25,
    batch_size=16,
    callbacks=callbacks
)

Epoch 1/25


  5/540 [..............................] - ETA: 2:00:12 - loss: 1.6794 - accuracy: 0.3250